In [ ]:
from neo4j import GraphDatabase
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()

In [2]:
df = pd.read_csv("clean_movies.csv")

In [ ]:
uri = os.getenv("uri")
username = os.getenv("username_neo")
password = os.getenv("password")

In [4]:
driver = GraphDatabase.driver(uri, auth=(username,password))

In [5]:
from neo4j import GraphDatabase

def insert_batch(tx, rows):
    tx.run("""
        UNWIND $rows AS row
        MERGE (m:Movie {name: row.movie})
        MERGE (d:Director {name: row.director})
        MERGE (m)-[:DIRECTED_BY]->(d)
    """, rows=rows)

    tx.run("""
        UNWIND $rows AS row
        MATCH (m:Movie {name: row.movie})
        UNWIND row.genres AS genre
        MERGE (g:Genre {name: genre})
        MERGE (m)-[:HAS_GENRE]->(g)
    """, rows=rows)

    tx.run("""
        UNWIND $rows AS row
        MATCH (m:Movie {name: row.movie})
        UNWIND row.companies AS company
        MERGE (c:Company {name: company})
        MERGE (m)-[:PRODUCED_BY]->(c)
    """, rows=rows)


def prepare_batch(df):
    batch = []
    for _, row in df.iterrows():
        batch.append({
            "movie": str(row["original_title"]).strip(),
            "director": str(row["director"]).strip(),
            "genres": [g.strip() for g in str(row["genres"]).split(",") if g.strip()],
            "companies": [c.strip() for c in str(row["production_companies"]).split(",") if c.strip()]
        })
    return batch


BATCH_SIZE = 500

batch = prepare_batch(df)

with driver.session() as session:
    for i in range(0, len(batch), BATCH_SIZE):
        chunk = batch[i:i + BATCH_SIZE]
        session.execute_write(insert_batch, chunk)
        print(f"Inserted rows {i+1} to {min(i+BATCH_SIZE, len(batch))}")

driver.close()
print("Done!")

Inserted rows 1 to 500
Inserted rows 501 to 1000
Inserted rows 1001 to 1500
Inserted rows 1501 to 2000
Inserted rows 2001 to 2500
Inserted rows 2501 to 3000
Inserted rows 3001 to 3500
Inserted rows 3501 to 4000
Inserted rows 4001 to 4500
Inserted rows 4501 to 4803
Done!
